# Searching and Matching Patterns in Files

This notebook covers how to apply Python's `re` module to search and match regular expression patterns inside files. You'll learn how to read files efficiently, find the first match, extract all matches, search for file names, and scan multiple files for product codes.

## Overview

Topics covered in this module:

- Search a file for a word (first match)
- Find file names in a file
- Search a file for patterns
- Search one and multiple files for product codes

## Why Read Files Line by Line?

Reading a file one line at a time is **more memory efficient** than loading the entire file into memory at once. It also allows you to process the file incrementally — which is especially important when the file could be large.

> **Warning:** Reading large files all at once can crash your program. If you can't control the file size, play it safe and read line by line.

For small files where you are certain of the size, reading the whole content at once is acceptable and is used in the examples below.

### Reading a File Line by Line

In [1]:
# Memory-friendly approach: one line at a time
# with open('file.txt') as file:
#     for line in file:
#         process(line)

### Reading a Small File All at Once

If you know the file is small, you can read it entirely into a string variable. This is the approach used in most examples below.

In [2]:
# with open('file.txt', 'r') as f:
#     text = f.read()

## Demo 1: Search a File to Find the First Match

**Goal:** Open a file, use `re.search()` to find the first occurrence of a word or pattern, and print the result.

**Steps:**
1. Open a file
2. Use the Python `search` function
3. Print the outcome

In [3]:
import re

# Create a sample file for demonstration
sample_text = """invoice number 1234
customer: Alice
product: widget
invoice date: 2024-01-15
total: 99.99
"""

with open('sample.txt', 'w') as f:
    f.write(sample_text)

In [4]:
import re

with open('sample.txt', 'r') as f:
    text = f.read()

match = re.search(r'invoice', text)

if match:
    print("Found:", match.group())
    print("At position:", match.start(), "-", match.end())
else:
    print("No match found")

Found: invoice
At position: 0 - 7


## Demo 2: Find File Names in a File

**Goal:** Build a regex pattern from a list of common file extensions and use `re.findall()` to extract all matching file names from a text file.

**Steps:**
1. Create a common extensions list
2. Read a file
3. Check for a regex pattern built using the extensions list
4. Find all the matches with the `findall` function

> Adjust the extensions list for your own requirements.

In [5]:
import re

# Sample file content containing file names
file_list_text = """Attached files: report.pdf, image.png, data.csv
Also see: notes.txt, archive.zip, presentation.pptx
Scripts: process.py, config.json
"""

with open('file_list.txt', 'w') as f:
    f.write(file_list_text)

In [6]:
import re

# Define common file extensions
extensions = ['pdf', 'png', 'csv', 'txt', 'zip', 'pptx', 'py', 'json']

# Build the pattern from the extensions list
ext_pattern = '|'.join(extensions)
pattern = r'\b\w+\.(?:' + ext_pattern + r')\b'

with open('file_list.txt', 'r') as f:
    text = f.read()

file_names = re.findall(pattern, text)
print("Found file names:", file_names)

Found file names: ['report.pdf', 'image.png', 'data.csv', 'notes.txt', 'archive.zip', 'presentation.pptx', 'process.py', 'config.json']


## Demo 3: Search a File for Patterns

**Goal:** Define multiple regex patterns, search for them in a file, and display the results using match objects.

**Steps:**
1. Define regex patterns
2. Search for them in the file
3. Display the result using the match objects

In [7]:
import re

# Sample data file with mixed content
mixed_text = """Order AB12-CD34 placed on 2024-03-15
Reference: 123-456
Customer email: alice@example.com
Order XY99-ZZ01 confirmed
Reference: 789-012
"""

with open('orders.txt', 'w') as f:
    f.write(mixed_text)

In [8]:
import re

# Define patterns to search for
patterns = {
    'order_code': r'[A-Z]{2}\d{2}-[A-Z]{2}\d{2}',
    'reference':  r'\b\d{3}-\d{3}\b',
    'date':       r'\d{4}-\d{2}-\d{2}',
}

with open('orders.txt', 'r') as f:
    text = f.read()

for name, pattern in patterns.items():
    matches = re.findall(pattern, text)
    print(f"{name}: {matches}")

order_code: ['AB12-CD34', 'XY99-ZZ01']
reference: ['123-456', '789-012']
date: ['2024-03-15']


## Demo 4: Find Custom Product Codes in a File

**Goal:** Read a file and extract all product codes matching a custom format.

**Steps:**
1. Read a file
2. Create the pattern
3. Use the `findall` function
4. Display the matches

The pattern below matches two formats:
- `AB12-CD34` — two uppercase letters, two digits, dash, two uppercase letters, two digits
- `123-456` — three digits, dash, three digits

In [9]:
import re

with open('sample.txt', 'r') as f:
    text = f.read()

pattern = r"\b[A-Z]{2}\d{2}-[A-Z]{2}\d{2}\b|\b\d{3}-\d{3}\b"

matches = re.findall(pattern, text)
print("All product codes:", matches)

All product codes: []


In [10]:
# More realistic demo: file with product codes
import re

product_text = """Inventory report
AB12-CD34 - in stock
XY99-ZZ01 - out of stock
123-456 - discontinued
789-012 - available
random text without a code
"""

with open('products.txt', 'w') as f:
    f.write(product_text)

with open('products.txt', 'r') as f:
    text = f.read()

pattern = r"\b[A-Z]{2}\d{2}-[A-Z]{2}\d{2}\b|\b\d{3}-\d{3}\b"
matches = re.findall(pattern, text)
print("All product codes:", matches)

All product codes: ['AB12-CD34', 'XY99-ZZ01', '123-456', '789-012']


## Demo 5: Find Custom Product Codes in Multiple Files

**Goal:** Scan several files for product codes and write all results to an output file.

**Steps:**
1. Wrap the product-code-finding logic in a function that also writes results to a file
2. Create a list of files to scan
3. Loop over the list and call the function for each file

In [11]:
import re

# Create a second sample file
product_text_2 = """Warehouse B
MN34-OP56 - pending
QR78-ST90 - shipped
321-654 - returned
"""
with open('products2.txt', 'w') as f:
    f.write(product_text_2)

In [12]:
import re

pattern = r"\b[A-Z]{2}\d{2}-[A-Z]{2}\d{2}\b|\b\d{3}-\d{3}\b"

def find_product_codes(filename, output_file):
    with open(filename, 'r') as f:
        text = f.read()
    matches = re.findall(pattern, text)
    with open(output_file, 'a') as out:
        out.write(f"\n--- {filename} ---\n")
        for code in matches:
            out.write(code + '\n')
    return matches

# List of files to scan
files_to_scan = ['products.txt', 'products2.txt']
output = 'found_codes.txt'

# Clear previous results
open(output, 'w').close()

for file in files_to_scan:
    codes = find_product_codes(file, output)
    print(f"{file}: {codes}")

print("\nResults written to:", output)

products.txt: ['AB12-CD34', 'XY99-ZZ01', '123-456', '789-012']
products2.txt: ['MN34-OP56', 'QR78-ST90', '321-654']

Results written to: found_codes.txt


## Summary

| Function | Description |
|---|---|
| `re.search(pattern, text)` | Returns the **first** match object, or `None` |
| `re.findall(pattern, text)` | Returns a **list of all** matching strings |
| `re.finditer(pattern, text)` | Returns an iterator of match objects |

**Key takeaways:**
- Read large files line by line to avoid memory issues.
- Use `re.findall()` to collect all matches in a file at once.
- Build regex patterns dynamically from lists (e.g. extensions, code formats).
- Wrap file-scanning logic in a function to reuse it across multiple files.
- Always adjust patterns to your specific data format.